In [10]:
import pandas as pd
import numpy as np
from datetime import datetime

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.metrics import roc_auc_score, precision_recall_curve, auc, average_precision_score
from sklearn.feature_selection import RFECV
from xgboost import XGBClassifier
from scipy.stats import ks_2samp
import matplotlib.pyplot as plt
import shap

from src.preprocessing import load_data, merge_data, change_dtypes
from src.modeling import DefaultModel

## 1. Preprocessing

We will start by 
- loading the data
- merging the two datasets
- changing the data types of the columns as needed
- dropping unnecessary columns and duplicate rows.

Reasoning:
- AGE column is the current age of the individual, but what we need is the age at the time of application. So, we will calculate the age by subtracting the birth year from the application year. But after calculating the age, we can see that everyone is either 51, 52 or 53 years old at the time of application, which doesn't provide much information for our model. 
- birth_year is 1970 for everyone, so it doesn't provide any information for our model.
- SPENDING is not relevant for our model as we are trying to predict DEFAULT based on the medical history of the individuals, not their spending habits after the application.
- APPLICATION_DATE is not relevant for our model.

In [11]:
df1, df2 = load_data(['data/interview homework file A.csv', 'data/interview homework file B.csv']) # Load the two datasets from the specified file paths
df = merge_data(df1, df2, on='ID') # Merge the two datasets on the 'ID' column after removing subcode '--xxx'
df = change_dtypes(df, str_cols=['ID'], date_cols=['APPLICATION_DATE', 'birth_year']).sort_values('APPLICATION_DATE') # Change the data types of specified columns
#df['AGE'] = df['APPLICATION_DATE'].dt.year - df['birth_year'].dt.year # Calculate the age of each individual by subtracting the birth year from the application year

df['AGE_APP'] = df['AGE'] - (datetime.now().year - df['APPLICATION_DATE'].dt.year)
cols_to_drop = ['birth_year', 'SPENDING', 'ANNUAL_INCOME', 'AGE'] #'APPLICATION_DATE'
df = df.drop_duplicates(subset='ID').drop(columns=cols_to_drop)

In [12]:
df['MAJORDRG_lag_concentration'] = df[[f'MAJORDRG_lag{i}' for i in range(1, 13)]].apply(lambda row: [row[f'MAJORDRG_lag{i}'] for i in range(1, 13)], axis=1).apply(lambda x: sum(x) / len([i for i in x if i > 0]) if any(i > 0 for i in x) else 0)
df['MINORDRG_lag_concentration'] = df[[f'MINORDRG_lag{i}' for i in range(1, 13)]].apply(lambda row: [row[f'MINORDRG_lag{i}'] for i in range(1, 13)], axis=1).apply(lambda x: sum(x) / len([i for i in x if i > 0]) if any(i > 0 for i in x) else 0)

In [27]:
# binning the age variable
bins = [0, 25, 35, 45, 55, 65, 75, np.inf]

df['AGE_APP'] = pd.cut(df['AGE_APP'], bins=bins, labels=[0, 1, 2, 3, 4, 5, 6], right=False)

In [28]:
df.groupby(['CARDHLDR']).apply(lambda x: len(x)/len(df))

CARDHLDR
0    0.219057
1    0.780943
dtype: float64

In [29]:
X = df.drop(columns=['CARDHLDR', 'ID', 'DEFAULT'])
y = df['DEFAULT']
approved_mask = df['CARDHLDR']


In [30]:
#X_train, X_test, y_train, y_test= train_test_split(X[approved_mask==1], y[approved_mask==1], test_size=0.2, stratify=y[approved_mask==1], random_state=42)

# time based split
cutoff_date = df[df['CARDHLDR']==1]['APPLICATION_DATE'].quantile(0.67)

train_mask = (df['APPLICATION_DATE'] <= cutoff_date) & (approved_mask == 1)
test_mask = (df['APPLICATION_DATE'] > cutoff_date) & (approved_mask == 1)

X_train, y_train = X[train_mask].drop(columns=['APPLICATION_DATE']), y[train_mask]
X_test, y_test = X[test_mask].drop(columns=['APPLICATION_DATE']), y[test_mask]


In [15]:
cutoff_date

Timestamp('2023-01-01 00:00:00')

In [ ]:
from sklearn.feature_selection import mutual_info_classif

mi = mutual_info_classif(X_train, y_train)
pd.Series(mi, index=X_train.columns).sort_values(ascending=False)

In [ ]:
correlations = X_train.corrwith(y_train).abs().sort_values(ascending=False)
print(correlations)

In [ ]:
X_train.head()

In [ ]:
scaler = StandardScaler()
cols_to_scale = ['ACADMOS', 'ADEPCNT', 'MONTHLY_INCOME']
X_train[cols_to_scale] = scaler.fit_transform(X_train[cols_to_scale])
X_test[cols_to_scale] = scaler.transform(X_test[cols_to_scale])
X[cols_to_scale] = scaler.transform(X[cols_to_scale])

In [ ]:
model = Model(model='logistic')
model.fit(X_train, y_train)
y_pred_proba = model.predict(X_test)
roc_auc, pr_auc, roc_curve, pr_curve = model.evaluate(y_test, y_pred_proba)


train_roc_auc = roc_auc_score(y_train, model.predict(X_train))
train_precision, train_recall, _ = precision_recall_curve(y_train, model.predict(X_train))
train_pr_auc = auc(train_recall, train_precision)
train_roc_auc, train_pr_auc, roc_auc, pr_auc

In [ ]:
coefs = pd.Series(model.model.coef_[0], index=X_train.columns).sort_values()
coefs

In [ ]:
model = Model(model='logistic', params={'class_weight': 'balanced'})
model.fit(X_train, y_train)
y_pred_proba = model.predict(X_test)
roc_auc, pr_auc, roc_curve, pr_curve = model.evaluate(y_test, y_pred_proba)

train_roc_auc = roc_auc_score(y_train, model.predict(X_train))
train_precision, train_recall, _ = precision_recall_curve(y_train, model.predict(X_train))
train_pr_auc = auc(train_recall, train_precision)
train_roc_auc, train_pr_auc, roc_auc, pr_auc

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
y_pred = (y_pred_proba >= 0.5).astype(int)
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

accuracy, precision, recall, f1

In [ ]:
model = Model(model='xgboost', params={'scale_pos_weight': 9, 'max_depth': 3, 'n_estimators': 100, 'learning_rate': 0.1, 'lambda': 1, 'alpha': 0})
#model.fit_weights(X, approved_mask)
model.fit(X_train, y_train)
y_pred_proba = model.predict(X_test)
roc_auc, pr_auc, roc_curve, pr_curve = model.evaluate(y_test, y_pred_proba)


train_roc_auc = roc_auc_score(y_train, model.predict(X_train))
train_precision, train_recall, _ = precision_recall_curve(y_train, model.predict(X_train))
train_pr_auc = auc(train_recall, train_precision)
train_roc_auc, train_pr_auc, roc_auc, pr_auc

In [ ]:
params = {
    'max_depth': 4,          # shallower trees
    'min_child_weight': 50,  # needs 50 samples to make a split
    'subsample': 0.7,        # row sampling
    'colsample_bytree': 0.7, # feature sampling
    'reg_alpha': 1.0,        # L1
    'reg_lambda': 5.0,       # L2
    'learning_rate': 0.01,
    'n_estimators': 500
}

model = Model(model='xgboost', params=params)
#model.fit_weights(X, approved_mask)
model.fit(X_train, y_train)
y_pred_proba = model.predict(X_test)
roc_auc, pr_auc, roc_curve, pr_curve = model.evaluate(y_test, y_pred_proba)


train_roc_auc = roc_auc_score(y_train, model.predict(X_train))
train_precision, train_recall, _ = precision_recall_curve(y_train, model.predict(X_train))
train_pr_auc = auc(train_recall, train_precision)
train_roc_auc, train_pr_auc, roc_auc, pr_auc

In [ ]:
rfecv = RFECV(estimator=XGBClassifier(), step=5, min_features_to_select=10, cv=5, scoring='roc_auc')
rfecv.fit(X_train, y_train)
optimal_features = X_train.columns[rfecv.support_]
optimal_features

In [ ]:
model = Model(model='xgboost', params={'scale_pos_weight': 9, 'max_depth': 3, 'n_estimators': 100, 'learning_rate': 0.05, 'lambda': 3, 'alpha': 0})
model.fit(X_train[optimal_features], y_train)
y_pred_proba = model.predict(X_test[optimal_features])
roc_auc, pr_auc, roc_curve, pr_curve = model.evaluate(y_test, y_pred_proba)


train_roc_auc = roc_auc_score(y_train, model.predict(X_train[optimal_features]))
train_precision, train_recall, _ = precision_recall_curve(y_train, model.predict(X_train[optimal_features]))
train_pr_auc = auc(train_recall, train_precision)
train_roc_auc, train_pr_auc, roc_auc, pr_auc

In [ ]:
feature_importances = pd.Series(model.model.feature_importances_, index=X_train[optimal_features].columns).sort_values(ascending=False)
feature_importances

In [ ]:
model = Model(model='xgboost')
model.fit(X_train, y_train)
y_pred_proba = model.predict(X_test)
roc_auc, pr_auc, roc_curve, pr_curve = model.evaluate(y_test, y_pred_proba)


train_roc_auc = roc_auc_score(y_train, model.predict(X_train))
train_precision, train_recall, _ = precision_recall_curve(y_train, model.predict(X_train))
train_pr_auc = auc(train_recall, train_precision)
train_roc_auc, train_pr_auc, roc_auc, pr_auc

In [ ]:
params = {
    'max_depth': 2,          # shallower trees
    'min_child_weight': 50,  # needs 50 samples to make a split
    'subsample': 0.7,        # row sampling
    'colsample_bytree': 0.7, # feature sampling
    'reg_alpha': 1.0,        # L1
    'reg_lambda': 5.0,       # L2
    'learning_rate': 0.01,
    'n_estimators': 500,
    'scale_pos_weight': 1,
}

model = DefaultModel(model='xgboost', params=params)
model.fit(X_train, y_train)
y_pred_proba = model.predict(X_test)
roc_auc, pr_auc, roc_curve, pr_curve = model.evaluate(y_test, y_pred_proba)


train_roc_auc = roc_auc_score(y_train, model.predict(X_train))
train_precision, train_recall, _ = precision_recall_curve(y_train, model.predict(X_train))
train_pr_auc = auc(train_recall, train_precision)
print(train_roc_auc, train_pr_auc, roc_auc, pr_auc)

In [ ]:
# feature importance
importance = pd.Series(automl.model.estimator.feature_importances_, index=X_train.columns).sort_values(ascending=False)

In [8]:
tscv = TimeSeriesSplit(n_splits=5)

model = DefaultModel(model_type='xgboost')

rfecv = RFECV(estimator=model._model_builder(), step=1, min_features_to_select=10, cv=tscv, scoring='average_precision')
rfecv.fit(X_train, y_train)

optimal_features = X_train.columns[rfecv.support_]
optimal_features

Index(['ACADMOS', 'ADEPCNT', 'MAJORDRG', 'MINORDRG', 'OWNRENT',
       'MONTHLY_INCOME', 'SELFEMPL', 'MAJORDRG_lag2', 'MAJORDRG_lag3',
       'MAJORDRG_lag12', 'MINORDRG_lag1', 'MINORDRG_lag2', 'AGE_APP',
       'MAJORDRG_lag_concentration'],
      dtype='str')

In [35]:
tscv = TimeSeriesSplit(n_splits=3)

model = DefaultModel(model_type='xgboost')

"""Best parameters: {'colsample_bytree': 0.3, 'learning_rate': 0.01, 'max_depth': 3, 
'min_child_weight': 50, 'n_estimators': 25, 
'reg_alpha': 2.0, 'reg_lambda': 1.5, 'scale_pos_weight': 10, 
'subsample': 0.5}"""

param_grid= {'colsample_bytree': [0.3], 
             'learning_rate': [0.01], 
             'max_depth': [3], 
             'min_child_weight': [75], 
             'n_estimators': [50], 
             'reg_alpha': [2.0], 
             'reg_lambda': [2], 
             'scale_pos_weight': [10], 
             'subsample': [0.5]}

grid_search = GridSearchCV(estimator=model, param_grid=param_grid, cv=tscv, scoring='roc_auc', n_jobs=-1, return_train_score=True)
grid_search.fit(X_train, y_train)

print("Best parameters:", grid_search.best_params_)
print("Best TEST ROC AUC:", grid_search.best_score_)
print("Best TRAIN ROC AUC:", grid_search.cv_results_['mean_train_score'][grid_search.best_index_])

best_estimator = grid_search.best_estimator_
y_pred_proba = best_estimator.predict_proba(X_test)[:,1]

print(f'PR AUC: {average_precision_score(y_test, y_pred_proba):.2f}')
print(f'ROC AUC: {roc_auc_score(y_test, y_pred_proba):.2f}')

Best parameters: {'colsample_bytree': 0.3, 'learning_rate': 0.01, 'max_depth': 3, 'min_child_weight': 75, 'n_estimators': 50, 'reg_alpha': 2.0, 'reg_lambda': 2, 'scale_pos_weight': 10, 'subsample': 0.5}
Best TEST ROC AUC: 0.5237395222445809
Best TRAIN ROC AUC: 0.6164656751775018
PR AUC: 0.11
ROC AUC: 0.51


In [39]:
feature_importances = pd.Series(best_estimator.model_.feature_importances_, index=X_train.columns).sort_values(ascending=False)
optimal_features = feature_importances[feature_importances > 0].index
optimal_features

Index(['MONTHLY_INCOME', 'MAJORDRG_lag_concentration', 'ACADMOS', 'MINORDRG',
       'AGE_APP', 'ADEPCNT', 'OWNRENT', 'MINORDRG_lag_concentration',
       'MAJORDRG_lag12', 'MAJORDRG', 'SELFEMPL', 'MINORDRG_lag12'],
      dtype='str')

Best parameters: {'colsample_bytree': 0.3, 'learning_rate': 0.01, 'max_depth': 3, 'min_child_weight': 50, 'n_estimators': 25, 'reg_alpha': 2.0, 'reg_lambda': 1.5, 'scale_pos_weight': 10, 'subsample': 0.5}

In [22]:
tscv = TimeSeriesSplit(n_splits=5)

scaler = StandardScaler()
cols_to_scale = ['ACADMOS', 'ADEPCNT', 'MONTHLY_INCOME', 'AGE_APP']
X_train[cols_to_scale] = scaler.fit_transform(X_train[cols_to_scale])
X_test[cols_to_scale] = scaler.transform(X_test[cols_to_scale])
X[cols_to_scale] = scaler.transform(X[cols_to_scale])

model = DefaultModel(model_type='logistic')

param_grid = {
    'C': [0.01, 0.1, 1, 10],
}

grid_search = GridSearchCV(estimator=model, param_grid=param_grid, cv=tscv, scoring='roc_auc', n_jobs=-1, return_train_score=True)
grid_search.fit(X_train[optimal_features], y_train)

print("Best parameters:", grid_search.best_params_)
print("Best TEST ROC AUC:", grid_search.best_score_)
print("Best TRAIN ROC AUC:", grid_search.cv_results_['mean_train_score'][grid_search.best_index_])

best_estimator = grid_search.best_estimator_
y_pred_proba = best_estimator.predict_proba(X_test[optimal_features])[:,1]

print(f'PR AUC: {average_precision_score(y_test, y_pred_proba):.2f}')
print(f'ROC AUC: {roc_auc_score(y_test, y_pred_proba):.2f}')

Best parameters: {'C': 1}
Best TEST ROC AUC: 0.5454191258910701
Best TRAIN ROC AUC: 0.5848891778327456
PR AUC: 0.11
ROC AUC: 0.52


In [40]:
tscv = TimeSeriesSplit(n_splits=3)

model = DefaultModel(model_type='xgboost')

param_grid = {
    'max_depth': [2, 3, 4],
    'min_child_weight': [10, 25, 50, 75],
    'subsample': [0.5, 0.7, 1.0],
    'colsample_bytree': [0.5],
    'reg_alpha': [2.0, 3.0, 4.0],
    'reg_lambda': [2.0, 3.0, 4.0],
    'learning_rate': [0.005, 0.01, 0.1],
    'n_estimators': [25, 50, 75, 100],
    'scale_pos_weight': [10]
}

grid_search = GridSearchCV(estimator=model, param_grid=param_grid, cv=tscv, scoring='roc_auc', n_jobs=-1, return_train_score=True)
grid_search.fit(X_train[optimal_features], y_train)

print("Best parameters:", grid_search.best_params_)
print("Best TEST ROC AUC:", grid_search.best_score_)
print("Best TRAIN ROC AUC:", grid_search.cv_results_['mean_train_score'][grid_search.best_index_])

best_estimator = grid_search.best_estimator_
y_pred_proba = best_estimator.predict_proba(X_test[optimal_features])[:,1]

print(f'PR AUC: {average_precision_score(y_test, y_pred_proba):.2f}')
print(f'ROC AUC: {roc_auc_score(y_test, y_pred_proba):.2f}')

Best parameters: {'colsample_bytree': 0.5, 'learning_rate': 0.1, 'max_depth': 2, 'min_child_weight': 50, 'n_estimators': 25, 'reg_alpha': 3.0, 'reg_lambda': 3.0, 'scale_pos_weight': 10, 'subsample': 0.5}
Best TEST ROC AUC: 0.5318252856152434
Best TRAIN ROC AUC: 0.6183420669283154
PR AUC: 0.11
ROC AUC: 0.51


In [ ]:
default_scores = y_pred_proba[y_test == 1]
nondefault_scores = y_pred_proba[y_test == 0]
# KS test
ks_stat, pval = ks_2samp(default_scores, nondefault_scores)
print(f"KS statistic: {ks_stat:.3f}")
print(f"p-value: {pval:.3f}")

In [ ]:
# shap values for best estimator using native xgboost explainer
import shap
explainer = shap.Explainer(best_estimator)
shap_values = explainer(X_test[optimal_features])
shap.summary_plot(shap_values, X_test[optimal_features])

In [ ]:
best_model = grid_search.best_estimator_

important_features = pd.Series(best_model.feature_importances_, index=X_train[optimal_features].columns).sort_values(ascending=False)
important_features
optimal_features2 = important_features[important_features > 0].index.tolist()

In [ ]:
optimal_features2

In [ ]:
# cv scores from sklearn import
from sklearn.model_selection import cross_validate

tscv = TimeSeriesSplit(n_splits=5)

params = {'colsample_bytree': 0.7, 'learning_rate': 0.05, 'max_depth': 2, 'min_child_weight': 250, 'n_estimators': 20, 'reg_alpha': 3.0, 'reg_lambda': 0.5, 'scale_pos_weight': 9, 'subsample': 0.7}

model = DefaultModel(model='xgboost', params=params)

cv_scores = cross_validate(model, X_train[optimal_features], y_train, cv=tscv, scoring='roc_auc', return_train_score=True)

print("Test ROC AUC scores:", cv_scores['test_score'])
print("Train ROC AUC scores:", cv_scores['train_score'])


In [ ]:
print("Best parameters:", grid_search.best_params_)
print("Best TEST PR AUC:", grid_search.best_score_)
print("Best TRAIN PR AUC:", grid_search.cv_results_['mean_train_score'][grid_search.best_index_])

In [ ]:
X_train

In [ ]:
default_scores = y_pred_proba[y_test==1]
nondefault_scores = y_pred_proba[y_test==0]

cdf_default = np.searchsorted(np.sort(default_scores), np.sort(y_pred_proba)) / len(default_scores)
cdf_nondefault = np.searchsorted(np.sort(nondefault_scores), np.sort(y_pred_proba)) / len(nondefault_scores)

plt.figure(figsize=(10, 6))
plt.plot(np.sort(y_pred_proba), cdf_default, label='CDF of Default Scores', color='red')
plt.plot(np.sort(y_pred_proba), cdf_nondefault, label='CDF of Non-Default Scores', color='blue')
plt.title('CDF of Predicted Probabilities')
plt.xlabel('Predicted Probability of Default')
plt.ylabel('Cumulative Probability')
plt.legend()
plt.show()

In [ ]:
from scipy.stats import ks_2samp

# KS test
ks_stat, pval = ks_2samp(default_scores, nondefault_scores)
print(f"KS statistic: {ks_stat:.3f}")
print(f"p-value: {pval:.3f}")

In [ ]:
from sklearn.model_selection import learning_curve
params = {
    'max_depth': 4,          # shallower trees
    'min_child_weight': 50,  # needs 50 samples to make a split
    'subsample': 0.7,        # row sampling
    'colsample_bytree': 0.7, # feature sampling
    'reg_alpha': 1.0,        # L1
    'reg_lambda': 5.0,       # L2
    'learning_rate': 0.01,
    'n_estimators': 500,
    'scale_pos_weight': 9,
}

model = Model(model='xgboost', params=params)


train_sizes, train_scores, test_scores = learning_curve(model.model, X_train, y_train, cv=5, scoring='average_precision', n_jobs=-1)

import matplotlib.pyplot as plt
plt.figure(figsize=(10, 6))
plt.plot(train_sizes, train_scores.mean(axis=1), label='Training Precision', marker='o')
plt.plot(train_sizes, test_scores.mean(axis=1), label='Validation Precision', marker='o')
plt.xlabel('Training Set Size')
plt.ylabel('Precision')
plt.title('Learning Curve')
plt.legend()
plt.show()

In [ ]:
precision, recall, _ = pr_curve
import matplotlib.pyplot as plt
plt.figure(figsize=(10, 6))
plt.plot( recall, precision, label='PR Curve')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.legend()
plt.show()

In [ ]:
feature_importances = pd.Series(model.model.feature_importances_, index=X_train.columns).sort_values(ascending=False)
feature_importances